# FIRMS Data collection (NASA FIRMS API) 
For our analysis we use wildfire data provieded by NASA's LANCE programm (Land, Atmosphere Near real-time Capability for Earth observation). The FIRMS dataset (Fire Information for Resource Management System) gives acess to MODIS data onboard Aqua & Terra satelites, as well as Visible Infrared Imaging Radiometer Suite (VIIRS) data aboard S-NPP, NOAA 20 and NOAA 21. In this analysis only VIIRS data of all three available satelites is used because of it's much better spatial resolution (~375 m).

The core idea of this whole project is to create a script which analyses the situation always on the foundation of the newest available data. Threrefore an API is used to download the most recent datasets of the mentioned sensors.

# Important: Time window can be adjusted!
Before running the script we can specify for how many days back (starting today) we want to analyse the wildfires in south-america. Defaut value is 1, e.g. we look at the last 24 hours. Only integers ranging from 1 to 7 are valid!

In [ ]:
# Set analysis time window
time_window = 1

# Be aware of the API rate limits when setting the time window. 
# A larger time window may result in more data points, which could lead to hitting the API rate limits.
# Consider as well to adjust the other two parameters (MAX_DISTANCE_KM and TIME_THRESHOLD) in the Analysis.ipynb notebook
# to better suit the chosen time window and the expected density of fire detections.

In [ ]:
import requests
import pandas as pd
import os
from io import StringIO
from datetime import datetime, timezone

# --------------------------------------------------
# NASA FIRMS API Settings
# --------------------------------------------------

MAP_KEY = "01b9b20f1c9e80d44560acd21f5a79a7"

# Bounding Box for South America
# west,south,east,north
AREA = "-85,-57,-32,14"

# --------------------------------------------------
# Sensors 
# --------------------------------------------------

SENSORS = [
    "VIIRS_NOAA21_NRT",
    "VIIRS_NOAA20_NRT",
    "VIIRS_SNPP_NRT"
]

#day range in days (1 = last 24 hours, 7 = last 7 days, etc.)
DAY_RANGE = time_window

# --------------------------------------------------
# Output folder
# --------------------------------------------------

OUTPUT_DIR = os.path.join("..", "data", "raw")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Starting download for all sensors...")

for SOURCE in SENSORS:

    print(f"Loading data for sensor: {SOURCE}")

    url = (
        f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        f"{MAP_KEY}/{SOURCE}/{AREA}/{DAY_RANGE}"
    )

    print(url)

    try:
        response = requests.get(url)
        response.raise_for_status()

        df = pd.read_csv(StringIO(response.text))

        print(f"Data points found ({SOURCE}): {len(df)}")

        timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')

        filename = f"firms_{SOURCE}_south_america_{timestamp}.csv"
        file_path = os.path.join(OUTPUT_DIR, filename)

        if not df.empty:
            df.to_csv(file_path, index=False)
            print(f"Data saved: {file_path}")
        else:
            print("No data found for this sensor.")

    except Exception as e:
        print(f"Error with {SOURCE}: {e}")

print("Finished.")


Starte Download für alle Sensoren...
Lade Daten für Sensor: VIIRS_NOAA21_NRT
https://firms.modaps.eosdis.nasa.gov/api/area/csv/01b9b20f1c9e80d44560acd21f5a79a7/VIIRS_NOAA21_NRT/-85,-57,-32,14/1
Datensätze gefunden (VIIRS_NOAA21_NRT): 1848
Gespeichert: ..\data\raw\firms_VIIRS_NOAA21_NRT_south_america_20260521_223946.csv
Lade Daten für Sensor: VIIRS_NOAA20_NRT
https://firms.modaps.eosdis.nasa.gov/api/area/csv/01b9b20f1c9e80d44560acd21f5a79a7/VIIRS_NOAA20_NRT/-85,-57,-32,14/1
Datensätze gefunden (VIIRS_NOAA20_NRT): 2013
Gespeichert: ..\data\raw\firms_VIIRS_NOAA20_NRT_south_america_20260521_223947.csv
Lade Daten für Sensor: VIIRS_SNPP_NRT
https://firms.modaps.eosdis.nasa.gov/api/area/csv/01b9b20f1c9e80d44560acd21f5a79a7/VIIRS_SNPP_NRT/-85,-57,-32,14/1
Datensätze gefunden (VIIRS_SNPP_NRT): 1995
Gespeichert: ..\data\raw\firms_VIIRS_SNPP_NRT_south_america_20260521_223949.csv
Fertig.
